# EDA des Behavior Trees XML

Ce notebook construit un pipeline d'analyse complet pour le dataset BTGenBot/Nav4Rails. Chaque entrée contient une consigne (`instruction`), une description en langage naturel (`input`) et un Behavior Tree XML (`output`).

Objectifs principaux :

1. Charger et auditer la qualité du dataset.
2. Parser les Behavior Trees XML avec `xml.etree.ElementTree`.
3. Extraire des métriques structurelles, syntaxiques et sémantiques.
4. Construire des tables `pandas` exploitables pour l'analyse.
5. Visualiser les distributions, motifs, attributs et graphes BT.
6. Explorer la relation entre texte naturel et structure XML.
7. Préparer les pistes avancées : Tree Edit Distance, clustering, embeddings de graphes, GNN/Tree-LSTM et future conversion JSONL.

Le dépôt originel BTGenBot est ajouté comme sous-module dans `external/BTGenBot` pour garder une référence traçable au projet publié : https://github.com/AIRLab-POLIMI/BTGenBot.git

In [3]:
from __future__ import annotations

import importlib.metadata as package_metadata
import json
import math
import re
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def parse_version_tuple(version: str) -> tuple[int, ...]:
    return tuple(int(part) for part in re.findall(r"\d+", version)[:3])


try:
    matplotlib_version = package_metadata.version("matplotlib")
    numpy_major = parse_version_tuple(np.__version__)[0]
    matplotlib_parts = parse_version_tuple(matplotlib_version)
    if numpy_major >= 2 and matplotlib_parts < (3, 8):
        raise ImportError(
            f"matplotlib {matplotlib_version} may be incompatible with NumPy {np.__version__}; "
            "upgrade matplotlib or use NumPy < 2 to enable plots."
        )
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTTING = True
except Exception as exc:
    plt = None
    sns = None
    HAS_PLOTTING = False
    PLOTTING_IMPORT_ERROR = exc

try:
    import networkx as nx
except ImportError:
    nx = None

try:
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.preprocessing import StandardScaler
except Exception:
    KMeans = None
    TfidfVectorizer = None
    cosine_similarity = None
    StandardScaler = None

if HAS_PLOTTING:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    print(f"Plotting disabled in this environment: {PLOTTING_IMPORT_ERROR}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_PATH = PROJECT_ROOT / "dataset" / "bt_dataset.json"
SUBMODULE_PATH = PROJECT_ROOT / "external" / "BTGenBot"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"BTGenBot submodule present: {SUBMODULE_PATH.exists()}")

Plotting disabled in this environment: matplotlib 3.5.1 may be incompatible with NumPy 2.2.6; upgrade matplotlib or use NumPy < 2 to enable plots.
Project root: /home/mlatoundji/studies/dev/nav4rails/repositories/BTGenBot_Nav4Rails
Dataset path: /home/mlatoundji/studies/dev/nav4rails/repositories/BTGenBot_Nav4Rails/dataset/bt_dataset.json
BTGenBot submodule present: True


## Fonctions utilitaires

Les fonctions ci-dessous gardent volontairement les transformations explicites : parsing XML strict, nettoyage minimal optionnel, parcours d'arbre, extraction de métriques, construction de tables et conversion en graphe. Le dataset contient plusieurs dialectes de Behavior Trees, donc les règles de catégorisation restent descriptives plutôt que normatives.

In [4]:
CONTROL_NODES = {
    "Sequence", "SequenceStar", "ReactiveSequence", "PipelineSequence", "Fallback",
    "ReactiveFallback", "Parallel", "RecoveryNode", "RoundRobin", "Root",
}
DECORATOR_NODES = {
    "RateController", "DistanceController", "SpeedController", "GoalUpdater",
    "RetryUntilSuccesful", "RetryUntilSuccessful", "Repeat", "Inverter", "ForceSuccess",
    "ForceFailure", "KeepRunningUntilFailure", "RunOnce", "Timeout",
}
NAVIGATION_ACTIONS = {
    "ComputePathToPose", "ComputePathThroughPoses", "FollowPath", "NavigateToPose",
    "NavigateThroughPoses", "ClearEntireCostmap", "ClearCostmapExceptRegion",
    "ClearCostmapAroundRobot", "Spin", "Wait", "BackUp", "DriveOnHeading",
}
MODEL_NODES = {"TreeNodesModel", "input_port", "output_port", "inout_port"}
STRUCTURAL_NODES = {"root", "BehaviorTree", "SubTree"}

KEYWORD_GROUPS = {
    "navigation": ["navigate", "navigation", "path", "goal", "planner", "controller", "move", "pose", "robot"],
    "recovery": ["recover", "recovery", "retry", "fallback", "failure", "clear", "costmap", "obstacle"],
    "object_manipulation": ["object", "pick", "place", "grasp", "bottle", "glass", "item", "arm"],
    "condition_logic": ["condition", "check", "if", "known", "valid", "state", "success", "failure"],
    "parallel_loop": ["parallel", "repeat", "loop", "continuous", "periodically", "cycle", "running"],
}


def load_dataset(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)
    if not isinstance(data, list):
        raise TypeError(f"Expected a list of records, got {type(data).__name__}")
    return data


def sanitize_xml(xml_text: str) -> str:
    """Minimal cleanup for recurrent XML defects without changing BT semantics."""
    cleaned = xml_text.strip()
    cleaned = re.sub(r"^\ufeff", "", cleaned)
    cleaned = re.sub(r"<\?xml\s+version\s*=\s*['\"]1\.0['\"]\s*\?>", '<?xml version="1.0"?>', cleaned)
    cleaned = re.sub(r"<!--(.*?)-->", lambda m: "<!--" + m.group(1).replace("--", "-") + "-->", cleaned, flags=re.S)
    return cleaned


def parse_xml(xml_text: str, sanitize: bool = False) -> tuple[ET.Element | None, str, str | None]:
    source = sanitize_xml(xml_text) if sanitize else xml_text
    try:
        return ET.fromstring(source), "parsed_sanitized" if sanitize else "parsed", None
    except ET.ParseError as exc:
        return None, "parse_error", str(exc)


def iter_nodes(root: ET.Element) -> list[tuple[ET.Element, int, str, str | None, int]]:
    rows: list[tuple[ET.Element, int, str, str | None, int]] = []
    stack = [(root, 0, "0", None, 0)]
    while stack:
        node, depth, path, parent_path, sibling_index = stack.pop()
        rows.append((node, depth, path, parent_path, sibling_index))
        children = list(node)
        for index, child in reversed(list(enumerate(children))):
            stack.append((child, depth + 1, f"{path}.{index}", path, index))
    return rows


def node_label(element: ET.Element) -> str:
    return element.attrib.get("ID") or element.attrib.get("name") or element.tag


def node_category(element: ET.Element) -> str:
    tag = element.tag
    node_id = element.attrib.get("ID", "")
    label = node_id or tag
    if tag in CONTROL_NODES:
        return "control"
    if tag in DECORATOR_NODES:
        return "decorator"
    if tag == "Action" or tag in NAVIGATION_ACTIONS or label in NAVIGATION_ACTIONS:
        return "action"
    if tag == "Condition":
        return "condition"
    if tag in MODEL_NODES:
        return "model"
    if tag in STRUCTURAL_NODES:
        return "structural"
    if tag == "SubTree":
        return "subtree"
    return "custom"


def to_float_or_nan(value: Any) -> float:
    try:
        return float(str(value).strip())
    except (TypeError, ValueError):
        return np.nan


def tree_metrics(record_id: int, root: ET.Element, xml_text: str, input_text: str) -> dict[str, Any]:
    nodes = iter_nodes(root)
    depths = [depth for _, depth, _, _, _ in nodes]
    child_counts = [len(list(element)) for element, *_ in nodes]
    width_by_depth = Counter(depths)
    n_nodes = len(nodes)
    n_edges = max(n_nodes - 1, 0)
    n_leaves = sum(1 for count in child_counts if count == 0)
    internal_counts = [count for count in child_counts if count > 0]
    max_depth = max(depths) if depths else 0
    avg_width = float(np.mean(list(width_by_depth.values()))) if width_by_depth else 0.0
    max_width = max(width_by_depth.values()) if width_by_depth else 0
    avg_branching = float(np.mean(internal_counts)) if internal_counts else 0.0
    leaf_ratio = n_leaves / n_nodes if n_nodes else 0.0
    complexity = n_nodes * math.log2(max_depth + 2) * (avg_branching + 1)
    tags = Counter(element.tag for element, *_ in nodes)
    categories = Counter(node_category(element) for element, *_ in nodes)
    return {
        "record_id": record_id,
        "n_nodes": n_nodes,
        "n_edges": n_edges,
        "max_depth": max_depth,
        "n_leaves": n_leaves,
        "leaf_ratio": leaf_ratio,
        "avg_width": avg_width,
        "max_width": max_width,
        "avg_branching_factor": avg_branching,
        "complexity_score": complexity,
        "n_behavior_trees": tags.get("BehaviorTree", 0),
        "n_subtrees": tags.get("SubTree", 0),
        "n_actions": categories.get("action", 0),
        "n_conditions": categories.get("condition", 0),
        "n_control": categories.get("control", 0),
        "n_decorators": categories.get("decorator", 0),
        "xml_length": len(xml_text),
        "input_length": len(input_text),
        "input_words": len(re.findall(r"\b\w+\b", input_text.lower())),
    }


def extract_node_rows(record_id: int, root: ET.Element) -> list[dict[str, Any]]:
    rows = []
    for element, depth, path, parent_path, sibling_index in iter_nodes(root):
        rows.append({
            "record_id": record_id,
            "node_path": path,
            "parent_path": parent_path,
            "sibling_index": sibling_index,
            "depth": depth,
            "tag": element.tag,
            "label": node_label(element),
            "category": node_category(element),
            "n_children": len(list(element)),
            "attrs": dict(element.attrib),
        })
    return rows


def extract_attribute_rows(record_id: int, root: ET.Element) -> list[dict[str, Any]]:
    rows = []
    for element, depth, path, _, _ in iter_nodes(root):
        for key, value in element.attrib.items():
            rows.append({
                "record_id": record_id,
                "node_path": path,
                "depth": depth,
                "tag": element.tag,
                "category": node_category(element),
                "attr_name": key,
                "attr_value": value,
                "attr_value_num": to_float_or_nan(value),
            })
    return rows


def extract_edge_rows(record_id: int, root: ET.Element) -> list[dict[str, Any]]:
    nodes = {path: element for element, _, path, _, _ in iter_nodes(root)}
    rows = []
    for child, depth, path, parent_path, sibling_index in iter_nodes(root):
        if parent_path is None:
            continue
        parent = nodes[parent_path]
        rows.append({
            "record_id": record_id,
            "parent_path": parent_path,
            "child_path": path,
            "depth": depth,
            "sibling_index": sibling_index,
            "parent_tag": parent.tag,
            "child_tag": child.tag,
            "parent_category": node_category(parent),
            "child_category": node_category(child),
            "edge_label": f"{parent.tag}->{child.tag}",
        })
    return rows


def subtree_signature(element: ET.Element, max_depth: int = 2) -> str:
    if max_depth <= 0 or len(list(element)) == 0:
        return element.tag
    children = ",".join(subtree_signature(child, max_depth - 1) for child in list(element))
    return f"{element.tag}({children})"


def extract_pattern_rows(record_id: int, root: ET.Element, max_signature_depth: int = 2) -> list[dict[str, Any]]:
    rows = []
    nodes = iter_nodes(root)
    by_path = {path: element for element, _, path, _, _ in nodes}
    parent_by_path = {path: parent_path for _, _, path, parent_path, _ in nodes}
    for element, depth, path, parent_path, _ in nodes:
        rows.append({
            "record_id": record_id,
            "pattern_type": "subtree_signature",
            "depth": depth,
            "pattern": subtree_signature(element, max_signature_depth),
        })
        if parent_path is not None:
            parent = by_path[parent_path]
            rows.append({
                "record_id": record_id,
                "pattern_type": "parent_child",
                "depth": depth,
                "pattern": f"{parent.tag}->{element.tag}",
            })
            grandparent_path = parent_by_path.get(parent_path)
            if grandparent_path is not None:
                grandparent = by_path[grandparent_path]
                rows.append({
                    "record_id": record_id,
                    "pattern_type": "grandparent_parent_child",
                    "depth": depth,
                    "pattern": f"{grandparent.tag}->{parent.tag}->{element.tag}",
                })
    return rows


def bt_to_networkx(record_id: int, root: ET.Element):
    if nx is None:
        raise ImportError("networkx is required for graph conversion")
    graph = nx.DiGraph(record_id=record_id)
    for element, depth, path, parent_path, sibling_index in iter_nodes(root):
        graph.add_node(
            path,
            tag=element.tag,
            label=node_label(element),
            category=node_category(element),
            depth=depth,
            sibling_index=sibling_index,
        )
        if parent_path is not None:
            graph.add_edge(parent_path, path)
    return graph


def add_keyword_features(df: pd.DataFrame, text_col: str = "input") -> pd.DataFrame:
    enriched = df.copy()
    lowered = enriched[text_col].fillna("").str.lower()
    for group, keywords in KEYWORD_GROUPS.items():
        pattern = r"\b(" + "|".join(re.escape(word) for word in keywords) + r")\b"
        enriched[f"kw_{group}"] = lowered.str.count(pattern)
        enriched[f"has_{group}"] = enriched[f"kw_{group}"] > 0
    return enriched

## Chargement et qualité XML

Cette étape produit une table brute des enregistrements et une table de statut de parsing. Les erreurs XML ne sont pas supprimées : elles sont conservées pour comprendre les limites du dataset et éviter un biais vers les arbres faciles à parser.

In [5]:
records = load_dataset(DATASET_PATH)

df_records = pd.DataFrame(records).reset_index(names="record_id")
for column in ["instruction", "input", "output"]:
    if column not in df_records:
        df_records[column] = ""

df_records["instruction_length"] = df_records["instruction"].fillna("").str.len()
df_records["input_length"] = df_records["input"].fillna("").str.len()
df_records["input_words"] = df_records["input"].fillna("").str.findall(r"\b\w+\b").str.len()
df_records["xml_length"] = df_records["output"].fillna("").str.len()
df_records["xml_lines"] = df_records["output"].fillna("").str.count("\n") + 1
df_records["is_duplicate_xml"] = df_records.duplicated("output", keep=False)
df_records["is_duplicate_input"] = df_records.duplicated("input", keep=False)
df_records = add_keyword_features(df_records)

parsed_roots: dict[int, ET.Element] = {}
parse_rows = []
for row in df_records.itertuples(index=False):
    root, status, error = parse_xml(row.output, sanitize=False)
    sanitized_used = False
    sanitized_error = None
    if root is None:
        root, sanitized_status, sanitized_error = parse_xml(row.output, sanitize=True)
        if root is not None:
            status = sanitized_status
            error = None
            sanitized_used = True
    if root is not None:
        parsed_roots[row.record_id] = root
    parse_rows.append({
        "record_id": row.record_id,
        "parse_status": status,
        "parse_error": error or sanitized_error,
        "sanitized_used": sanitized_used,
        "is_parsed": root is not None,
    })

df_parse = pd.DataFrame(parse_rows)
df_records = df_records.merge(df_parse, on="record_id", how="left")

print(f"Records: {len(df_records)}")
print(f"Parsed XML records: {df_records['is_parsed'].sum()} / {len(df_records)}")
display(df_records["parse_status"].value_counts(dropna=False).rename_axis("parse_status").reset_index(name="count"))
display(df_records.loc[~df_records["is_parsed"], ["record_id", "parse_error", "input", "output"]].head(10))
display(df_records.head(3))

Records: 594
Parsed XML records: 578 / 594


,parse_status,count
0,parsed,578
1,parse_error,16


,record_id,parse_error,input,output
5,5,"not well-formed (invalid token): line 2, column 10",The behavior tree represents the decision-making process for a robot. The robot is programmed to perform a series of tasks based on certain conditions and a...,"<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
6,6,"not well-formed (invalid token): line 2, column 10",The behavior tree represents a robot's task to assist in serving a drink. The robot is programmed to perform a series of actions based on certain conditions...,"<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
47,47,"not well-formed (invalid token): line 2, column 10",The behavior tree outlines a task for a robot to perform in a home environment. The robot is programmed to execute a series of actions in a fallback sequenc...,"<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
89,89,"not well-formed (invalid token): line 2, column 10",The behavior tree outlines the decision-making process for a robot. The robot is programmed to perform various tasks based on certain conditions and actions...,"<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
118,118,"mismatched tag: line 73, column 6",The behavior tree outlines a sequence of tasks for a robotic system. The robot is programmed to perform the following tasks in sequence: \n1. Push a puck\n2...,"<!-- Evitement\n<ReactiveSequence>\n<Fallback>\n <Condition ID=""NoObstacle""/>\n <Avoid/>\n</Fallback>\n<Move pos= positions['P2'] />\n</ReactiveSequen..."
211,211,"not well-formed (invalid token): line 2, column 10",The behavior tree represents the decision-making process for a robot. The robot is programmed to perform a series of actions based on certain conditions and...,"<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
212,212,"not well-formed (invalid token): line 2, column 12","The behavior tree outlines the decision-making process for a robot tasked with interacting with a ball and a bin. The robot follows a sequence of actions, a...","<root main_tree_to_execute=""BehaviorTree"">\n <!-- ----------------------------------- -->\n <BehaviorTree ID=""BehaviorTree"">\n <Fallback name..."
261,261,"not well-formed (invalid token): line 2, column 10","The behavior tree represents a robot's task. The robot is programmed to perform a series of actions based on certain conditions and events. Initially, it wi...","<root main_tree_to_execute=""BehaviorTree"">\n <!--------------------------------------->\n <BehaviorTree ID=""BehaviorTree"">\n <Root>\n ..."
273,273,"mismatched tag: line 72, column 2","The behavior tree orchestrates a robotic task involving multiple sub-tasks. Initially, the robot checks the quantity of specific objects and then proceeds w...","<?xml version=""1.0""?>\n<root main_tree_to_execute ='Main_Tree'>\n\t<BehaviourTree ID = 'Main_Tree'>\n\t\t<Fallback>\n\t\t\t<ReactiveSequence name = 'root_se..."
291,291,"not well-formed (invalid token): line 6, column 56","The behavior tree orchestrates a robot's movement based on visual input. The robot is programmed to continuously execute a sequence of movements, alternatin...","<root main_tree_to_execute = ""MainTree"">\n <BehaviorTree ID=""MainTree"">\n <Sequence>\n <SetBlackboard output_key=""Foward"" value=""i;0.5""..."


,record_id,instruction,input,output,instruction_length,input_length,input_words,xml_length,xml_lines,is_duplicate_xml,is_duplicate_input,kw_navigation,has_navigation,kw_recovery,has_recovery,kw_object_manipulation,has_object_manipulation,kw_condition_logic,has_condition_logic,kw_parallel_loop,has_parallel_loop,parse_status,parse_error,sanitized_used,is_parsed
0,0,"You will be provided a summary of a task performed by a behavior tree, and your objective is to express this behavior tree in XML format.","The behavior tree orchestrates the navigation of a robot by periodically replanning its global path at a frequency of 1 Hz. It utilizes a pipeline sequence,...","<!--\n This Behavior Tree replans the global path periodically at 1 Hz.\n-->\n\n<root main_tree_to_execute=""MainTree"">\n <BehaviorTree ID=""MainTree"">\n ...",137,536,83,448,15,True,False,12,True,0,False,0,False,0,False,1,True,parsed,None,False,True
1,1,"You will be provided a summary of a task performed by a behavior tree, and your objective is to express this behavior tree in XML format.",The behavior tree is designed to control a robot's navigation with continuous replanning of the global path. It uses a pipeline sequence that consists of tw...,"<!--\n This Behavior Tree replans the global path after every 1m.\n-->\n\n<root main_tree_to_execute=""MainTree"">\n <BehaviorTree ID=""MainTree"">\n <Pipe...",137,544,92,456,15,True,False,15,True,0,False,0,False,0,False,1,True,parsed,None,False,True
2,2,"You will be provided a summary of a task performed by a behavior tree, and your objective is to express this behavior tree in XML format.",The behavior tree is a simple sequential task for a robot. It first instructs the robot to move to a specific point (Go_point) and then to interact with a p...,"<root main_tree_to_execute = ""MainTree"" >\n <BehaviorTree ID=""MainTree"">\n <Sequence name=""root_sequence"">\n <Go_point name=""Go_point"" ...",137,316,51,284,9,False,False,4,True,0,False,2,True,0,False,0,False,parsed,None,False,True


## Extraction des métriques et tables analytiques

Les tables produites ici sont les bases réutilisables du pipeline :

- `df_bt_metrics` : une ligne par Behavior Tree parsé.
- `df_nodes` : une ligne par noeud XML.
- `df_attrs` : une ligne par attribut XML.
- `df_edges` : une ligne par relation parent-enfant.
- `df_patterns` : signatures de sous-arbres, bigrammes et trigrammes structurels.

In [6]:
metric_rows = []
node_rows = []
attribute_rows = []
edge_rows = []
pattern_rows = []

for record_id, root in parsed_roots.items():
    source = df_records.loc[df_records["record_id"] == record_id].iloc[0]
    metric_rows.append(tree_metrics(record_id, root, source["output"], source["input"]))
    node_rows.extend(extract_node_rows(record_id, root))
    attribute_rows.extend(extract_attribute_rows(record_id, root))
    edge_rows.extend(extract_edge_rows(record_id, root))
    pattern_rows.extend(extract_pattern_rows(record_id, root, max_signature_depth=2))

df_bt_metrics = pd.DataFrame(metric_rows).merge(
    df_records[[
        "record_id", "parse_status", "input", "output",
        "kw_navigation", "kw_recovery", "kw_object_manipulation",
        "kw_condition_logic", "kw_parallel_loop",
    ]],
    on="record_id",
    how="left",
)
df_nodes = pd.DataFrame(node_rows)
df_attrs = pd.DataFrame(attribute_rows)
df_edges = pd.DataFrame(edge_rows)
df_patterns = pd.DataFrame(pattern_rows)

print("Shapes")
display(pd.DataFrame({
    "table": ["df_records", "df_bt_metrics", "df_nodes", "df_attrs", "df_edges", "df_patterns"],
    "rows": [len(df_records), len(df_bt_metrics), len(df_nodes), len(df_attrs), len(df_edges), len(df_patterns)],
}))

display(df_bt_metrics.describe(include="number").T)
display(df_nodes["category"].value_counts().rename_axis("category").reset_index(name="count"))

Shapes


,table,rows
0,df_records,594
1,df_bt_metrics,578
2,df_nodes,21522
3,df_attrs,28102
4,df_edges,20944
5,df_patterns,62221


,count,mean,std,min,25%,50%,75%,max
record_id,578.0,297.816609,171.720151,0.0,149.250000,298.500000,445.750000,593.000000
n_nodes,578.0,37.235294,56.060873,3.0,9.000000,16.000000,36.000000,399.000000
n_edges,578.0,36.235294,56.060873,2.0,8.000000,15.000000,35.000000,398.000000
max_depth,578.0,5.396194,2.538532,2.0,3.250000,5.000000,6.000000,21.000000
n_leaves,578.0,23.797578,39.560219,1.0,4.000000,9.000000,23.000000,255.000000
leaf_ratio,578.0,0.528790,0.152666,0.2,0.400000,0.515909,0.649263,0.892857
avg_width,578.0,5.376697,7.465081,1.0,1.517857,2.571429,4.968750,54.750000
max_width,578.0,16.150519,26.672476,1.0,3.000000,6.000000,13.750000,172.000000
avg_branching_factor,578.0,2.270053,1.086578,1.0,1.500000,2.000000,2.722222,9.000000
complexity_score,578.0,457.204266,817.979838,12.0,60.845613,150.857143,403.557612,5649.231810


,category,count
0,action,6119
1,model,4568
2,control,3119
3,custom,2630
4,structural,2293
5,condition,1856
6,decorator,937


## Analyse structurelle et distributions

Ces vues donnent une première lecture de la taille, profondeur, largeur et complexité des BT. Elles permettent aussi d'identifier des familles d'arbres : très simples, très profonds, très larges, ou dominés par des modèles de noeuds.

In [7]:
structure_cols = [
    "n_nodes", "max_depth", "n_leaves", "avg_width", "max_width",
    "avg_branching_factor", "complexity_score", "xml_length", "input_words",
]

if HAS_PLOTTING:
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    for ax, col in zip(axes.ravel(), structure_cols):
        sns.histplot(df_bt_metrics[col], kde=True, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=df_bt_metrics, x="input_words", y="n_nodes", hue="max_depth", palette="viridis", ax=axes[0])
    axes[0].set_title("Taille du BT vs longueur de description")
    sns.scatterplot(data=df_bt_metrics, x="n_nodes", y="complexity_score", hue="n_control", palette="magma", ax=axes[1])
    axes[1].set_title("Complexité structurelle vs nombre de noeuds")
    plt.tight_layout()
    plt.show()

    corr_cols = [col for col in structure_cols + ["n_actions", "n_conditions", "n_control", "n_decorators"] if col in df_bt_metrics]
    plt.figure(figsize=(12, 9))
    sns.heatmap(df_bt_metrics[corr_cols].corr(numeric_only=True), cmap="vlag", center=0, annot=True, fmt=".2f")
    plt.title("Corrélations entre métriques structurelles")
    plt.tight_layout()
    plt.show()
else:
    print("Plotting skipped: matplotlib/seaborn are unavailable in this environment.")

display(df_bt_metrics.sort_values("complexity_score", ascending=False).head(10)[[
    "record_id", "n_nodes", "max_depth", "n_leaves", "avg_branching_factor",
    "complexity_score", "input"
]])

Plotting skipped: matplotlib/seaborn are unavailable in this environment.


,record_id,n_nodes,max_depth,n_leaves,avg_branching_factor,complexity_score,input
385,398,399,19,220,2.223464,5649.231810,"The behavior tree orchestrates a robot's interactions with a human player, offering various games and responding to the player's actions. It begins by initi..."
500,515,348,8,255,3.731183,5469.393870,This behavior tree orchestrates the decision-making and actions of a character in a simulated environment. The character is programmed to perform various ta...
540,556,345,8,252,3.698925,5385.274078,The behavior tree encompasses a complex set of tasks for an AI-controlled entity. It includes various sub-trees for different activities such as handling al...
136,141,363,8,247,3.120690,4968.974409,"The behavior tree represents a complex set of tasks for a robot. It includes various sub-trees and actions such as moving towards specific goals, interactin..."
242,249,289,8,210,3.645570,4459.919741,"The behavior tree outlines a set of tasks for an AI entity. The tasks include combat, acquiring a uniform, managing hunger and thirst, performing various jo..."
350,363,289,8,210,3.645570,4459.919741,The behavior tree orchestrates the actions of a character in a simulated environment. The character is programmed to perform various tasks based on its need...
15,17,289,8,210,3.645570,4459.919741,The behavior tree orchestrates the actions of a character in a simulated environment. The character's behaviors are governed by various conditions and actio...
407,420,289,8,210,3.645570,4459.919741,The behavior tree represents a complex set of tasks for an AI character. The main task involves the character performing various jobs and activities based o...
212,219,240,13,160,2.987500,3738.894300,The behavior tree represents a decision-making process for an autonomous vehicle. The vehicle's actions are determined by various conditions and sequences w...
479,494,196,19,134,3.145161,3568.545375,The behavior tree represents a decision-making process for an autonomous vehicle. The vehicle's actions are determined by various conditions and actions def...


## Vocabulaire XML, attributs et motifs récurrents

Cette section répond aux questions sur la diversité des opérateurs, la distribution des actions/conditions/contrôleurs, les paramètres XML et les sous-structures fréquentes.

In [8]:
top_tags = df_nodes["tag"].value_counts().head(30).rename_axis("tag").reset_index(name="count")
top_labels = df_nodes["label"].value_counts().head(30).rename_axis("label").reset_index(name="count")
top_attrs = df_attrs["attr_name"].value_counts().head(30).rename_axis("attr_name").reset_index(name="count")

display(top_tags.head(15))
display(top_labels.head(15))
display(top_attrs.head(15))

category_by_depth = (
    df_nodes.groupby(["depth", "category"])
    .size()
    .reset_index(name="count")
    .pivot(index="depth", columns="category", values="count")
    .fillna(0)
)

if HAS_PLOTTING:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    sns.barplot(data=top_tags, y="tag", x="count", ax=axes[0])
    axes[0].set_title("Top tags XML")
    sns.barplot(data=top_labels, y="label", x="count", ax=axes[1])
    axes[1].set_title("Top labels ID/name/tag")
    sns.barplot(data=top_attrs, y="attr_name", x="count", ax=axes[2])
    axes[2].set_title("Top attributs XML")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.heatmap(category_by_depth, cmap="Blues")
    plt.title("Catégories de noeuds par profondeur")
    plt.tight_layout()
    plt.show()
else:
    print("Plotting skipped: showing tabular summaries only.")

display(category_by_depth.head(20))

interesting_attrs = ["planner_id", "controller_id", "hz", "distance", "num_cycles", "number_of_retries", "service_name", "goal", "path"]
for attr_name in interesting_attrs:
    values = df_attrs.loc[df_attrs["attr_name"] == attr_name]
    if values.empty:
        continue
    print(f"\nAttribute: {attr_name} ({len(values)} occurrences)")
    display(values["attr_value"].value_counts().head(15).rename_axis("value").reset_index(name="count"))
    numeric_values = values["attr_value_num"].dropna()
    if HAS_PLOTTING and len(numeric_values) >= 5:
        plt.figure(figsize=(8, 3))
        sns.histplot(numeric_values, kde=True)
        plt.title(f"Distribution numérique: {attr_name}")
        plt.tight_layout()
        plt.show()

pattern_summary = (
    df_patterns.groupby(["pattern_type", "pattern"])
    .agg(count=("record_id", "size"), n_records=("record_id", "nunique"))
    .reset_index()
    .sort_values(["pattern_type", "count"], ascending=[True, False])
)

for pattern_type in ["parent_child", "grandparent_parent_child", "subtree_signature"]:
    print(f"\nTop patterns: {pattern_type}")
    display(pattern_summary.loc[pattern_summary["pattern_type"] == pattern_type].head(20))

,tag,count
0,Action,5482
1,input_port,3137
2,Condition,1856
3,Sequence,1460
4,BehaviorTree,942
5,output_port,897
6,SubTree,773
7,Fallback,658
8,root,578
9,SetBlackboard,364


,label,count
0,Sequence,1012
1,root,606
2,Fallback,492
3,SetBlackboard,347
4,MainTree,346
5,Move,259
6,TreeNodesModel,226
7,output,222
8,BehaviorTree,195
9,ComputePathToPose,188


,attr_name,count
0,ID,9158
1,name,6828
2,type,1692
3,default,970
4,main_tree_to_execute,578
5,value,421
6,output_key,364
7,goal,349
8,path,349
9,service_name,317


Plotting skipped: showing tabular summaries only.


category,action,condition,control,custom,decorator,model,structural
depth,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,578.0
1,0.0,0.0,0.0,21.0,0.0,226.0,942.0
2,2782.0,1033.0,756.0,160.0,86.0,4.0,300.0
3,550.0,113.0,624.0,874.0,310.0,4338.0,167.0
4,745.0,210.0,570.0,424.0,160.0,0.0,97.0
5,933.0,144.0,362.0,294.0,154.0,0.0,112.0
6,375.0,87.0,245.0,227.0,116.0,0.0,22.0
7,315.0,57.0,153.0,197.0,43.0,0.0,34.0
8,158.0,55.0,96.0,73.0,18.0,0.0,17.0


...

,pattern_type,pattern,count,n_records
2066,parent_child,TreeNodesModel->Action,2766,224
1326,parent_child,Action->input_port,2753,134
1778,parent_child,Sequence->Action,1166,202
2067,parent_child,TreeNodesModel->Condition,1033,139
2078,parent_child,root->BehaviorTree,942,577
1327,parent_child,Action->output_port,864,105
1991,parent_child,SequenceStar->Action,837,47
1476,parent_child,Fallback->Sequence,465,109
1371,parent_child,BehaviorTree->Sequence,402,292
1818,parent_child,Sequence->Condition,335,73



Top patterns: grandparent_parent_child


,pattern_type,pattern,count,n_records
1314,grandparent_parent_child,root->TreeNodesModel->Action,2766,224
1265,grandparent_parent_child,TreeNodesModel->Action->input_port,2753,134
1315,grandparent_parent_child,root->TreeNodesModel->Condition,1033,139
1266,grandparent_parent_child,TreeNodesModel->Action->output_port,864,105
390,grandparent_parent_child,Fallback->Sequence->Action,404,64
1306,grandparent_parent_child,root->BehaviorTree->Sequence,402,292
472,grandparent_parent_child,Fallback->SequenceStar->Action,379,17
1318,grandparent_parent_child,root->TreeNodesModel->SubTree,295,72
1264,grandparent_parent_child,TreeNodesModel->Action->inout_port,293,39
1040,grandparent_parent_child,Sequence->Fallback->Sequence,288,64



Top patterns: subtree_signature


,pattern_type,pattern,count,n_records
2088,subtree_signature,Action,4024,242
4452,subtree_signature,input_port,3137,138
2594,subtree_signature,Condition,1713,129
4456,subtree_signature,output_port,897,106
4249,subtree_signature,SubTree,742,126
4232,subtree_signature,SetBlackboard,364,72
4451,subtree_signature,inout_port,308,39
2098,subtree_signature,Action(input_port),281,100
2587,subtree_signature,ClearEntireCostmap,180,51
4287,subtree_signature,TextToSpeechActionClient,156,26


## Représentation graphe des Behavior Trees

Les BT sont naturellement des arbres orientés. `networkx` permet de les convertir en graphes pour inspecter les degrés, visualiser des exemples et préparer des embeddings ou modèles GNN.

In [13]:
if nx is None:
    print("networkx is not installed; skipping graph conversion and drawing.")
else:
    graph_rows = []
    graphs = {}
    for record_id, root in parsed_roots.items():
        graph = bt_to_networkx(record_id, root)
        graphs[record_id] = graph
        degrees = [degree for _, degree in graph.degree()]
        graph_rows.append({
            "record_id": record_id,
            "graph_nodes": graph.number_of_nodes(),
            "graph_edges": graph.number_of_edges(),
            "avg_degree": float(np.mean(degrees)) if degrees else 0.0,
            "max_degree": max(degrees) if degrees else 0,
            "n_roots": sum(1 for node in graph.nodes if graph.in_degree(node) == 0),
            "n_graph_leaves": sum(1 for node in graph.nodes if graph.out_degree(node) == 0),
        })
    df_graph_metrics = pd.DataFrame(graph_rows)
    display(df_graph_metrics.describe().T)

    example_ids = [
        int(df_bt_metrics.sort_values("n_nodes").iloc[0]["record_id"]),
        int(df_bt_metrics.iloc[(df_bt_metrics["n_nodes"] - df_bt_metrics["n_nodes"].median()).abs().argsort().iloc[0]]["record_id"]),
        int(df_bt_metrics.sort_values("n_nodes", ascending=False).iloc[0]["record_id"]),
    ]
    example_titles = ["Petit BT", "BT médian", "Grand BT"]

    if HAS_PLOTTING:
        for record_id, title in zip(example_ids, example_titles):
            graph = graphs[record_id]
            if graph.number_of_nodes() > 80:
                print(f"{title} record_id={record_id}: {graph.number_of_nodes()} noeuds, trop grand pour un rendu lisible complet.")
                continue
            try:
                pos = nx.nx_agraph.graphviz_layout(graph, prog="dot")
            except Exception:
                pos = nx.spring_layout(graph, seed=42)
            labels = {node: graph.nodes[node]["tag"] for node in graph.nodes}
            colors = pd.Series({node: graph.nodes[node]["category"] for node in graph.nodes}).astype("category").cat.codes
            plt.figure(figsize=(14, 7))
            nx.draw(graph, pos, labels=labels, node_color=colors, cmap=plt.cm.Set3, node_size=900, font_size=8, arrows=True)
            plt.title(f"{title} - record_id={record_id}")
            plt.axis("off")
            plt.show()
    else:
        print("Graph drawing skipped because plotting is unavailable.")

networkx is not installed; skipping graph conversion and drawing.


## Relation entre description naturelle et structure XML

Cette section explore si certains mots-clés ou thèmes textuels correspondent à des structures BT : taille de l'arbre, présence de contrôleurs, patterns de recovery, navigation, manipulation d'objets ou logique conditionnelle.

In [10]:
keyword_cols = [col for col in df_records.columns if col.startswith("kw_")]
text_structure = df_bt_metrics.merge(
    df_records[["record_id", "input", *keyword_cols]],
    on="record_id",
    how="left",
    suffixes=("", "_record"),
)

keyword_corr = (
    text_structure[keyword_cols + ["n_nodes", "max_depth", "n_actions", "n_conditions", "n_control", "complexity_score"]]
    .corr(numeric_only=True)
    .loc[keyword_cols, ["n_nodes", "max_depth", "n_actions", "n_conditions", "n_control", "complexity_score"]]
)
keyword_presence = df_records[[col.replace("kw_", "has_") for col in keyword_cols]].mean().sort_values(ascending=False)

display(keyword_corr)
display(keyword_presence.rename("presence_ratio").reset_index().rename(columns={"index": "keyword_group"}))

if HAS_PLOTTING:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(keyword_corr, cmap="vlag", center=0, annot=True, fmt=".2f", ax=axes[0])
    axes[0].set_title("Corrélation mots-clés texte vs structure")
    sns.barplot(x=keyword_presence.values, y=keyword_presence.index, ax=axes[1])
    axes[1].set_title("Part des descriptions contenant chaque thème")
    axes[1].set_xlabel("Ratio")
    plt.tight_layout()
    plt.show()

    for col in keyword_cols:
        group_col = col.replace("kw_", "has_")
        if group_col not in df_records:
            continue
        tmp = df_bt_metrics.merge(df_records[["record_id", group_col]], on="record_id", how="left")
        if tmp[group_col].nunique() < 2:
            continue
        plt.figure(figsize=(9, 4))
        sns.boxplot(data=tmp, x=group_col, y="complexity_score")
        plt.title(f"Complexité selon le thème {group_col}")
        plt.tight_layout()
        plt.show()
else:
    print("Plotting skipped: showing keyword correlation tables only.")

if TfidfVectorizer is None or KMeans is None or StandardScaler is None:
    print("scikit-learn is not installed; skipping TF-IDF and clustering.")
else:
    vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=500)
    text_matrix = vectorizer.fit_transform(df_bt_metrics["input"].fillna(""))

    structural_features = df_bt_metrics[[
        "n_nodes", "max_depth", "n_leaves", "avg_branching_factor",
        "n_actions", "n_conditions", "n_control", "n_decorators",
    ]].fillna(0)
    scaled_structural = StandardScaler().fit_transform(structural_features)

    k = min(5, len(df_bt_metrics))
    text_clusters = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(text_matrix)
    structure_clusters = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(scaled_structural)
    df_clusters = df_bt_metrics[["record_id", "input", *structural_features.columns]].copy()
    df_clusters["text_cluster"] = text_clusters
    df_clusters["structure_cluster"] = structure_clusters

    display(df_clusters.groupby("text_cluster")[structural_features.columns].mean().round(2))
    display(df_clusters.groupby("structure_cluster")[structural_features.columns].mean().round(2))

    cluster_crosstab = pd.crosstab(df_clusters["text_cluster"], df_clusters["structure_cluster"])
    display(cluster_crosstab)
    if HAS_PLOTTING:
        plt.figure(figsize=(8, 6))
        sns.heatmap(cluster_crosstab, annot=True, fmt="d", cmap="Blues")
        plt.title("Correspondance clusters texte vs clusters structure")
        plt.tight_layout()
        plt.show()

    if cosine_similarity is not None:
        n_sample = min(150, len(df_bt_metrics))
        sample = df_clusters.sample(n_sample, random_state=42)
        sample_idx = sample.index.to_numpy()
        text_sim = cosine_similarity(text_matrix[sample_idx])
        struct_sim = cosine_similarity(scaled_structural[sample_idx])
        upper = np.triu_indices_from(text_sim, k=1)
        sim_corr = np.corrcoef(text_sim[upper], struct_sim[upper])[0, 1]
        print(f"Corrélation similarité texte / similarité structure sur {n_sample} BT: {sim_corr:.3f}")

    display(df_clusters.head(10))

,n_nodes,max_depth,n_actions,n_conditions,n_control,complexity_score
kw_navigation,-0.196658,0.140637,-0.203747,-0.228124,-0.071277,-0.207245
kw_recovery,-0.075645,0.298466,-0.033622,-0.075251,0.099610,-0.088264
kw_object_manipulation,0.043589,-0.011881,0.015713,-0.027906,0.032679,0.041069
kw_condition_logic,0.058815,0.305946,0.076944,0.207426,0.144560,0.061688
kw_parallel_loop,-0.081251,0.150755,-0.105381,-0.129290,-0.055933,-0.095375


,keyword_group,presence_ratio
0,has_navigation,0.850168
1,has_condition_logic,0.609428
2,has_recovery,0.331650
3,has_parallel_loop,0.294613
4,has_object_manipulation,0.223906


Plotting skipped: showing keyword correlation tables only.
scikit-learn is not installed; skipping TF-IDF and clustering.


## Pistes avancées et recherche

### Tree Edit Distance

Comparer deux BT via une distance d'édition d'arbres permettrait d'identifier des familles structurelles indépendamment du texte. Une implémentation possible consiste à convertir chaque XML en arbre de labels (`tag` ou `tag:ID`) puis à utiliser une bibliothèque dédiée comme `zss` ou `apted`.

### Embeddings de graphes

Les tables `df_nodes` et `df_edges` préparent des graphes exploitables pour :

- Weisfeiler-Lehman graph kernels.
- Node2Vec/Graph2Vec.
- Encodage par chemins racine-feuille.
- GNN supervisé ou non supervisé.

### Tree-LSTM ou GNN

Pour une tâche aval de génération ou validation, chaque BT peut être représenté par :

- des features de noeuds : `tag`, `category`, `ID`, attributs normalisés ;
- des arêtes parent-enfant ordonnées ;
- un texte associé encodé par TF-IDF ou embeddings de phrases ;
- une cible : famille de BT, validité XML, complexité, opérateurs présents.

### Adaptation prompt et conversion JSONL

La future conversion JSONL pourra enrichir chaque exemple avec des métadonnées EDA : complexité, tags dominants, motifs, attributs Nav2, catégorie textuelle et statut de parsing. Cela permettra de filtrer les exemples trop bruités, équilibrer les familles structurelles et construire des prompts plus contrôlés.

In [11]:
summary = {
    "n_records": len(df_records),
    "n_parsed": int(df_records["is_parsed"].sum()),
    "n_parse_errors": int((~df_records["is_parsed"]).sum()),
    "median_nodes": float(df_bt_metrics["n_nodes"].median()),
    "median_depth": float(df_bt_metrics["max_depth"].median()),
    "top_tags": df_nodes["tag"].value_counts().head(10).to_dict(),
    "top_attributes": df_attrs["attr_name"].value_counts().head(10).to_dict(),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

# Optional exports for downstream experiments. Uncomment when persistent artifacts are needed.
# output_dir = PROJECT_ROOT / "analysis_outputs"
# output_dir.mkdir(exist_ok=True)
# df_records.to_csv(output_dir / "records_quality.csv", index=False)
# df_bt_metrics.to_csv(output_dir / "bt_metrics.csv", index=False)
# df_nodes.to_csv(output_dir / "bt_nodes.csv", index=False)
# df_attrs.to_csv(output_dir / "bt_attributes.csv", index=False)
# df_edges.to_csv(output_dir / "bt_edges.csv", index=False)
# df_patterns.to_csv(output_dir / "bt_patterns.csv", index=False)

{
  "n_records": 594,
  "n_parsed": 578,
  "n_parse_errors": 16,
  "median_nodes": 16.0,
  "median_depth": 5.0,
  "top_tags": {
    "Action": 5482,
    "input_port": 3137,
    "Condition": 1856,
    "Sequence": 1460,
    "BehaviorTree": 942,
    "output_port": 897,
    "SubTree": 773,
    "Fallback": 658,
    "root": 578,
    "SetBlackboard": 364
  },
  "top_attributes": {
    "ID": 9158,
    "name": 6828,
    "type": 1692,
    "default": 970,
    "main_tree_to_execute": 578,
    "value": 421,
    "output_key": 364,
    "goal": 349,
    "path": 349,
    "service_name": 317
  }
}
